# Manual Price Push

Push manual price overrides for specific SKUs using market data or fixed prices.

## How to use
1. Run cells 1-3 (setup + data loading)
2. Edit the `PUSH_LIST` in cell 4 with your product IDs and desired actions
3. Run cell 4 — prices are computed per product × cohort (using cohort-specific market data)
4. Review the output table
5. Set `MODE = 'live'` and run cell 5 to push

> Each product is automatically expanded to all 9 cohorts. Market-based actions use cohort-specific prices.  
> Main/general cohorts (695, 61, 699, 697, 698, 696) are auto-mirrored by the push handler.

## Available Actions
| Action | Description |
|--------|-------------|
| `market_min` | Lowest market price (P0 / minimum) |
| `market_25` | 25th percentile market price |
| `market_50` | Median market price (P50) |
| `market_75` | 75th percentile market price |
| `market_max` | Highest market price (maximum) |
| `market_avg` | Average of min and max market prices |
| `target_margin` | Price from brand-category target margin |
| `step_up` | Next tier above current price in the price list |
| `step_down` | Next tier below current price in the price list |
| `<number>` | Fixed price (e.g. `115`, `23.5`) |

> Only SKUs with stock > 0 are pushed.

In [11]:
# =============================================================================
# SETUP
# =============================================================================
import sys, os
import pandas as pd
import numpy as np
import pytz
from datetime import datetime

sys.path.insert(0, os.path.abspath('.'))
import setup_environment_2
setup_environment_2.initialize_env()

from db import query_snowflake
from constants import WAREHOUSE_MAPPING, COHORT_IDS

os.chdir(os.path.join(os.path.dirname(os.path.abspath('__file__')) or os.getcwd(), 'modules'))
%run push_prices_handler.ipynb
os.chdir('..')

CAIRO_TZ = pytz.timezone('Africa/Cairo')
TODAY = datetime.now(CAIRO_TZ).strftime('%Y-%m-%d')
print(f'Ready. Today: {TODAY}')

/home/ec2-user/.Renviron
/home/ec2-user/service_account_key.json
Push Prices Handler loaded at 2026-04-27 14:54:27 Cairo time
✓ API credentials loaded successfully
✓ Google Sheets client initialized
Ready. Today: 2026-04-27


In [12]:
# =============================================================================
# LOAD MARKET DATA + WAC + CURRENT PRICES + PACKING UNITS
# =============================================================================
os.chdir('modules')
%run market_data_module_2.ipynb
os.chdir('..')

print('Loading market data (V2)...')
market_data = get_market_data_v2()
print(f'  Market data: {len(market_data)} rows')

print('Loading WAC...')
WAC_QUERY = f'''
SELECT product_id, wac_p
FROM finance.all_cogs c
WHERE wac_p > 0
    AND CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP()) 
        BETWEEN c.from_date AND c.to_date
'''
df_wac = query_snowflake(WAC_QUERY)
print(f'  WAC: {len(df_wac)} rows')

print('Loading current prices...')
PRICES_QUERY = f'''

    SELECT 
        cohort_id,
        pu.product_id,
        pu.packing_unit_id,
        pu.basic_unit_count,
        AVG(cpu.price) AS current_price
    FROM cohort_product_packing_units cpu
    JOIN packing_unit_products pu ON pu.id = cpu.product_packing_unit_id
    WHERE cpu.cohort_id IN ({','.join(str(c) for c in COHORT_IDS)})
        AND cpu.created_at::DATE <> '2023-07-31'
        AND cpu.is_customized = TRUE
        and basic_unit_count = 1 
    GROUP BY 1,2,3,4

'''
df_prices = query_snowflake(PRICES_QUERY)
#df_prices = setup_environment_2.dwh_pg_query(PRICES_QUERY, columns=['cohort_id','product_id','packing_unit_id','basic_unit_count','current_price'])
print(f'  Prices: {len(df_prices)} rows')

print('Loading packing units...')
PU_QUERY = '''
WITH sales_check AS (
    SELECT DISTINCT
        pso.product_id,
        pso.packing_unit_id,
        SUM(pso.total_price) AS nmv
    FROM product_sales_order pso
    JOIN sales_orders so ON so.id = pso.sales_order_id
    WHERE so.created_at >= CURRENT_DATE - 60 
        AND so.sales_order_status_id NOT IN (7, 12)
        AND so.channel IN ('telesales', 'retailer')
        AND pso.purchased_item_count <> 0
    GROUP BY 1, 2
)
SELECT product_id, packing_unit_id, basic_unit_count
FROM (
    SELECT *, MAX(nmv) OVER (PARTITION BY product_id, is_basic_unit) AS top_nmv
    FROM (
        SELECT 
            pup.product_id,
            pup.packing_unit_id,
            pup.basic_unit_count,
            pup.is_basic_unit,
            COUNT(DISTINCT CASE WHEN pup.basic_unit_count = 1 THEN pup.packing_unit_id END) 
                OVER (PARTITION BY pup.product_id) AS total_basic,
            COALESCE(sc.nmv, 0) AS nmv
        FROM packing_unit_products pup
        LEFT JOIN sales_check sc 
            ON sc.product_id = pup.product_id 
            AND sc.packing_unit_id = pup.packing_unit_id
        WHERE pup.deleted_at IS NULL
    )
)
WHERE nmv = top_nmv OR (top_nmv = 0 AND total_basic <= 1)
'''
df_pus = query_snowflake(PU_QUERY)
print(f'  Packing units: {len(df_pus)} rows')

print('Loading product info...')
PRODUCT_QUERY = '''
SELECT p.id AS product_id,
       CONCAT(p.name_ar, ' ', p.size, ' ', product_units.name_ar) AS sku,
       b.name_ar AS brand,
       c.name_ar AS cat
FROM products p
JOIN brands b ON b.id = p.brand_id
JOIN categories c ON c.id = p.category_id
JOIN product_units ON product_units.id = p.unit_id
'''
df_products = query_snowflake(PRODUCT_QUERY)
print(f'  Products: {len(df_products)} rows')

print('Loading target margins...')
MARGIN_QUERY = f'''
SELECT brand, cat, target_margin
FROM (
    SELECT b.name_ar AS brand, c.name_ar AS cat, cplan.margin AS target_margin,
           ROW_NUMBER() OVER (PARTITION BY b.name_ar, c.name_ar ORDER BY cplan.date DESC) AS rn
    FROM performance.commercial_targets cplan
    JOIN brands b ON b.id = cplan.brand_id
    JOIN categories c ON c.id = cplan.category_id
    WHERE cplan.margin IS NOT NULL
    QUALIFY CASE 
        WHEN DATE_TRUNC('month', MAX(date) OVER()) = DATE_TRUNC('month', CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE) 
        THEN DATE_TRUNC('month', CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE)
        ELSE DATE_TRUNC('month', CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE - INTERVAL '1 month') 
    END = DATE_TRUNC('month', date)
)
WHERE rn = 1
'''
df_targets = query_snowflake(MARGIN_QUERY)
print(f'  Target margins: {len(df_targets)} rows')

print('Loading stocks...')
STOCKS_QUERY = '''
WITH parent_whs AS (
    SELECT * FROM (VALUES (236,343),(1,467),(962,343)) x(parent_id,child_id)
)
SELECT warehouse_id, product_id,
    CASE WHEN stocks_child IS NOT NULL AND stocks = 0 THEN stocks_child ELSE stocks END AS stocks
FROM (
    SELECT pw.warehouse_id, pw.product_id,
        pw.available_stock::INTEGER AS stocks,
        pw2.available_stock::INTEGER AS stocks_child
    FROM product_warehouse pw
    LEFT JOIN parent_whs p ON p.parent_id = pw.warehouse_id
    LEFT JOIN product_warehouse pw2 ON pw2.warehouse_id = p.child_id AND pw.product_id = pw2.product_id
    WHERE pw.warehouse_id NOT IN (6, 9, 10)
        AND pw.is_basic_unit = 1
)
'''
df_stocks = query_snowflake(STOCKS_QUERY)
print(f'  Stocks: {len(df_stocks)} rows')

print('\nAll data loaded.')

/home/ec2-user/.Renviron
/home/ec2-user/service_account_key.json
/home/ec2-user/.Renviron
/home/ec2-user/service_account_key.json
Queries Module | Timezone: America/Los_Angeles
✅ UTH and Last Hour functions defined
✅ Yesterday closing stock function defined
Fixed price & cart rule functions defined
get_commercial_min_prices() defined
get_commercial_price_ups() defined
get_margin_boundaries_region() defined
get_margin_boundaries_global() defined

QUERIES MODULE READY

Live Data Functions:
  • get_current_stocks()
  • get_packing_units()
  • get_current_prices()
  • get_current_wac()
  • get_current_cart_rules()

UTH Performance Functions:
  • get_uth_performance()         - UTH qty/retailers (Snowflake)
  • get_hourly_distribution()     - Historical hour contributions (Snowflake)
  • get_last_hour_performance()   - Last hour qty/retailers (DWH)

Stock Functions:
  • get_yesterday_closing_stock() - Yesterday closing stock with parent WH mapping

Override Functions:
  • get_fixed_prices()

In [13]:
# =============================================================================
# BUILD LOOKUP TABLE (product_id → market prices + WAC + target margin)
# =============================================================================

# V2 market_data returns (product_id, region, price_tiers, wac_p, target_margin, num_sources, market_data_source)
# Expand to cohorts for lookup
df_market_cohorts = expand_to_cohorts(market_data)

# Lookup is per (product_id, cohort_id)
lookup = df_products.merge(df_wac, on='product_id', how='left')

cohort_df = pd.DataFrame({'cohort_id': COHORT_IDS})
lookup = lookup.merge(cohort_df, how='cross')
lookup = lookup.merge(
    df_market_cohorts[['product_id', 'cohort_id', 'price_tiers', 'target_margin']],
    on=['product_id', 'cohort_id'], how='left'
)
lookup['price_tiers'] = lookup['price_tiers'].apply(lambda x: x if isinstance(x, list) else [])

# Fill target_margin for rows without V2 data
lookup = lookup.merge(df_targets, on=['brand', 'cat'], how='left', suffixes=('', '_tgt'))
for col in ['target_bm', 'target_bm_tgt', 'cat_target_margin', 'cat_target_margin_tgt']:
    if col in lookup.columns:
        lookup['target_margin'] = lookup['target_margin'].fillna(lookup[col])
lookup['target_margin'] = lookup['target_margin'].fillna(0.05)
lookup.drop(columns=[c for c in lookup.columns if c.startswith('target_bm') or c.startswith('cat_target')], inplace=True, errors='ignore')

# Derive market percentile prices from price_tiers list
def tiers_to_percentiles(tiers):
    if not tiers or len(tiers) == 0:
        return pd.Series({'market_min': np.nan, 'market_25': np.nan, 'market_50': np.nan,
                          'market_75': np.nan, 'market_max': np.nan, 'market_avg': np.nan})
    arr = np.array(tiers)
    return pd.Series({
        'market_min': arr[0],
        'market_25': np.percentile(arr, 25),
        'market_50': np.percentile(arr, 50),
        'market_75': np.percentile(arr, 75),
        'market_max': arr[-1],
        'market_avg': (arr[0] + arr[-1]) / 2,
    })

lookup = pd.concat([lookup, lookup['price_tiers'].apply(tiers_to_percentiles)], axis=1)

# Merge current base-unit price per (product_id, cohort_id)
base_prices = df_prices[df_prices['basic_unit_count'] == 1][['cohort_id', 'product_id', 'current_price']].drop_duplicates()
lookup = lookup.merge(base_prices, on=['product_id', 'cohort_id'], how='left')

# Merge stocks (sum across warehouses per product for display; warehouse-level for push filtering)
wh_df = pd.DataFrame(WAREHOUSE_MAPPING, columns=['region', 'warehouse', 'warehouse_id', 'cohort_id'])
product_stocks = df_stocks.groupby('product_id')['stocks'].sum().reset_index().rename(columns={'stocks': 'total_stocks'})
lookup = lookup.merge(product_stocks, on='product_id', how='left')
lookup['total_stocks'] = lookup['total_stocks'].fillna(0).astype(int)

print(f'Lookup table: {len(lookup)} rows (product x cohort)')
print(f'  With market data: {(lookup["price_tiers"].apply(len) > 0).sum()}')
print(f'  With WAC: {lookup["wac_p"].notna().sum()}')
print(f'  With target margin: {lookup["target_margin"].notna().sum()}')

Lookup table: 234329 rows (product x cohort)
  With market data: 44660
  With WAC: 76163
  With target margin: 234329


In [14]:
lookup[(lookup['total_stocks']>0)].to_excel('manual_data.xlsx')

In [ ]:
#lookup.to_excel('manual_data.xlsx')

In [ ]:
#lookup[(lookup['brand']=='ماريو')&(lookup['total_stocks']>0)]

In [16]:
# =============================================================================
# DEFINE YOUR SKU ACTIONS HERE
# =============================================================================
# Format: (product_id, action)
#   action can be: 'market_min', 'market_25', 'market_50', 'market_75',
#                  'market_max', 'market_avg', 'target_margin',
#                  'step_up', 'step_down', or a number (fixed price)
#
# step_up: next tier above current price in the price_tiers list
# step_down: next tier below current price in the price_tiers list
# Each product is automatically expanded to ALL cohorts.
# Market-based actions use the cohort-specific market price.
# Only SKUs with stock > 0 are pushed.

PUSH_LIST = [
(3985,704,36.8025457936824),
(25698,701,35.3815682215049),
(6779,700,81.0789473684211),
(24242,1124,22.8100016931385),
(13598,1124,84.8684210526316),
(2715,700,318.535145793966),
(25697,701,162.986207182957),
(25697,703,162.986207182957),
(25697,1123,162.986207182957),
(25697,1124,162.986207182957),
(25697,1125,162.986207182957),
(25697,1126,162.986207182957),
(5969,700,28.0713378097517),
(2270,701,43.1823567304649),
(22586,701,58.5323953399269),
(24142,700,568.420922164268),
(21320,703,10.9241133808185),
(25581,700,42.6315789473684),
(8905,701,21.7522784619239),
(25579,701,21.7894736842105),
(25578,700,44.5263157894737),
(25694,703,162.984392662901),
(4001,703,37.7752456763849),
(21703,704,88.5070154442953),
(26402,704,46.5),
(26401,704,46.5),
(25576,700,50.2105263157895),
(25700,700,130.285746389188),
(25700,701,130.285746389188),
(25700,703,130.285746389188),
(25700,704,130.285746389188),
(25700,1123,130.285746389188),
(25700,1125,130.285746389188),
(25700,1126,130.285746389188),
(13395,701,57.7864473684211),
(6692,701,235.747936359924),
(10716,703,312.321290137309),
(10716,704,312.321290137309),
(8721,701,19.2647368421053),
(1454,701,365.048881610873),
(22891,703,47.8955589492756),
(6194,701,84.1578947368421),
(25695,701,162.982072453706),
(25695,703,162.982072453706),
(25695,1123,162.982072453706),
(25695,1126,162.982072453706),
(3143,1124,91.2348589107816),
(26798,700,101.052631578947),
(1457,700,120.987970667246),
(25645,701,20.3497071031565),
(6798,700,190.027913346647),
(25860,703,336.436842105263),
(21713,703,27.44),
(3987,704,17.610155283229),
(26397,704,27.9),
(26396,704,27.9),
(26398,704,27.9),
(26399,704,27.9),
(25699,700,131.394254023172),
(25699,701,131.394254023172),
(25699,702,131.394254023172),
(25699,703,131.394254023172),
(25699,704,131.394254023172),
(25699,1123,131.394254023172),
(25699,1124,131.394254023172),
(24243,702,22.81),
(24243,704,22.81),
(3989,704,17.6507967647147),
(2310,704,18.3552631578947),
(3984,703,17.9779027593719),
(2718,701,270.142934382963),
(23526,700,433.00166028603),
(25508,704,97.4251155125906),
(12003,704,32.4902478670728),
(25508,702,96.1684200057846),
(12027,704,102.158894767168),
(9835,1126,203.959812276307),
(26553,700,600),
(2049,700,'market_min'),
(2049,701,'market_min'),
(1309,702,'market_min'),
(1309,702,'market_min'),
(10825,700,'market_min'),
(2109,701,'market_min'),
(6935,701,'market_min'),
(161,700,'market_min'),
(2188,704,'market_min'),
(97,1123,'market_min'),
(593,701,'market_min'),
(130,701,'market_min'),
(151,700,'market_min'),
(6936,700,'market_min'),
(4372,703,'market_min'),
(11681,703,'market_min'),
(11681,704,'market_min'),
(141,1126,'market_min'),
(25958,701,'market_min'),
(11685,702,'market_min'),
(11685,703,'market_min'),
(11685,704,'market_min'),
(11685,1126,'market_min'),
(205,700,'market_25'),
(205,701,'market_min'),
(10406,1126,'market_min'),
(7004,700,'market_25'),
(7004,701,'market_25'),
(7004,1123,'market_min'),
(145,700,'market_min'),
(432,1124,'market_min'),
(990,700,'market_25'),
(7005,703,'market_min'),
(25960,1126,'market_min'),
(143,700,'market_min'),
(149,703,'market_25'),
(149,1125,'market_25'),
(149,1126,'market_25'),
(9085,701,'market_min'),
(9085,704,'market_min'),
(9085,1124,'market_min'),
(10411,1124,'market_min'),
(147,700,'market_25'),
(147,701,'market_25'),
(147,1125,'market_25'),
(147,1126,'market_25'),
(10993,700,'market_min'),
(2805,701,'market_min'),
(10530,701,'market_min'),
(10816,703,'market_min'),
(10816,704,'market_min'),
(11737,700,'market_min'),
(10409,704,'market_min'),
(21704,701,'market_min'),
(21704,1123,'market_min'),
(21704,1125,'market_min'),
(13249,700,'market_min'),
(307,703,'market_min'),
(307,1126,'market_min'),
(26126,703,'market_min'),
(413,701,'market_min'),
(148,700,'market_25'),
(148,702,'market_25'),
(148,1125,'market_min'),
(148,1126,'market_min'),
(11684,702,'market_min'),
(11684,704,'market_min'),
(11684,1125,'market_min'),
(11684,1126,'market_min'),
(2368,1124,'market_min'),
(3840,1123,'market_min'),
(7523,700,'market_min'),
(439,1125,'market_min'),
(439,1126,'market_min'),
(23039,1126,'market_min'),
(5490,701,'market_min'),
(24705,703,'market_min'),
(4043,704,'market_25'),
(4043,1124,'market_min'),
(11739,701,'market_min'),
(8217,701,'market_min'),
(2331,704,'market_min'),
(2331,1124,'market_min'),
(23238,1123,'market_min'),
(23238,1126,'market_min'),
(2835,701,'market_min'),
(12496,700,'market_25'),
(10397,1126,'market_min'),
(2821,704,'market_min'),
(9910,1126,'market_min'),
(6629,703,'market_25'),
(6629,1124,'market_min'),
(506,702,'market_25'),
(506,1124,'market_25'),
(506,1126,'market_min'),
(11887,1126,'market_min'),
(9087,1124,'market_min'),
(6036,701,'market_min'),
(6036,704,'market_min'),
(3987,701,'market_min'),
(9840,703,'market_25'),
(22920,704,'market_25'),
(6630,703,'market_min'),
(4076,701,'market_min'),
(10815,702,'market_min'),
(21459,703,'market_min'),
(21459,704,'market_min'),
(21459,1124,'market_min'),
(11738,701,'market_min'),
(11738,1124,'market_min'),
(21699,704,'market_min'),
(11425,700,'market_min'),
(26128,700,'market_min'),
(26128,701,'market_min'),
(26128,702,'market_min'),
(26128,703,'market_min'),
(26128,704,'market_min'),
(26128,1124,'market_min'),
(3989,1123,'market_min'),
(8155,701,'market_min'),
(14041,704,'market_min'),
(12269,704,'market_min'),
(12269,1125,'market_min'),
(14017,701,'market_25'),
(23599,701,'market_min'),
(22037,701,'market_25'),
(11719,700,'market_min'),
(11719,704,'market_min'),
(13498,703,'market_25'),
(13498,1124,'market_min'),
(14038,1123,'market_min'),
(10581,703,'market_min'),
(13108,700,'market_min'),
(13108,703,'market_min'),
(9088,701,'market_min'),
(9088,704,'market_min'),
(9088,1124,'market_min'),
(19964,702,'market_min'),
(1407,1125,'market_25'),
(10275,703,'market_25'),
(19970,703,'market_min'),
(19966,1125,'market_min'),
(23546,704,'market_min'),
(21319,701,'market_min'),
(12993,702,'market_25'),
(12993,1125,'market_min'),
(2369,1124,'market_min'),
(2369,1125,'market_min'),
(858,702,'market_min'),
(3976,701,'market_min'),
(3976,702,'market_min'),
(936,701,'market_min'),
(2738,701,'market_min'),
(25709,701,'market_25'),
(19047,701,'market_min'),
(13264,703,'market_25'),
(13264,704,'market_min'),
(3028,701,'market_min'),
(12270,702,'market_min'),
(11721,700,'market_min'),
(11721,702,'market_min'),
(3984,701,'market_min'),
(21693,702,'market_min'),
(24670,704,'market_25'),
(3434,701,'market_25'),
(3434,702,'market_min'),
(3434,1124,'market_min'),
(3434,1125,'market_min'),
(3434,1126,'market_min'),
(11610,703,'market_min'),
(13036,704,'market_25'),
(21700,704,'market_min'),
(221,704,'market_25'),
(3582,701,'market_min'),
(3582,704,'market_min'),
(22634,700,'market_25'),
(22038,701,'market_25'),
(21468,701,'market_min'),
(21468,704,'market_min'),
(21468,1123,'market_min'),
(21468,1124,'market_min'),
(21468,1125,'market_min'),
(21476,704,'market_min'),
(9830,703,'market_min'),
(11586,703,'market_min'),
(21478,701,'market_min'),
(21478,702,'market_min'),
(13256,701,'market_min'),
(24245,704,'market_min'),
(24245,1126,'market_min'),
(5646,703,'market_25'),
(8686,703,'market_min'),
(3583,703,'market_min'),
(2099,704,'market_min'),
(19965,700,'market_min'),
(19965,701,'market_min'),
(8191,700,'market_min'),
(12493,704,'market_min'),
(109,703,'market_min'),
(9396,703,'market_min'),
(3072,701,'market_min'),
(3072,703,'market_min'),
(3072,704,'market_min'),
(27214,701,'market_min'),
(27214,704,'market_min'),
(8189,704,'market_min'),
(2492,702,'market_min'),
(7520,704,'market_min'),
(26125,701,'market_min'),
(26125,704,'market_min'),
(26127,702,'market_min'),
(26127,703,'market_min'),
(1705,704,'market_25'),
(10181,704,'market_25'),
(2212,701,'market_min'),
(2212,703,'market_25'),
(21474,703,'market_min'),
(12992,702,'market_25'),
(12992,703,'market_25'),
(26124,703,'market_min'),
(26124,704,'market_min'),
(11587,703,'market_min'),
(11587,1123,'market_min'),
(11788,701,'market_min'),
(1668,702,'market_min'),
(3985,701,'market_min'),
(11155,1126,'market_25'),
(20208,701,'market_min'),
(6540,703,'market_25'),
(11678,703,'market_min'),
(11856,1125,'market_min'),
(10278,703,'market_min'),
(3047,701,'market_min'),
(27212,704,'market_min'),
(3071,703,'market_min'),
(8907,701,'market_min'),
(3848,701,'market_min'),
(1324,703,'market_min'),
(1324,1126,'market_min'),
(21461,701,'market_min'),
(8872,701,'market_min'),
(8872,704,'market_min'),
(3988,703,'market_min'),
(3988,704,'market_min'),
(2308,701,'market_25'),
(2308,704,'market_25'),
(2111,1126,'market_25'),
(24718,701,'market_min'),
(22879,701,'market_min'),
(27216,701,'market_min'),
(27216,704,'market_min'),
(13855,704,'market_min'),
(150,702,'market_25'),
(150,704,'market_25'),
(150,1124,'market_25'),
(150,1125,'market_25'),
(2324,700,'market_min'),
(8190,700,'market_25'),
(27215,704,'market_min'),
(4715,701,'market_25'),
(13109,701,'market_min'),
(11968,701,'market_min'),
(8050,703,'market_min'),
(2190,702,'market_min'),
(12906,701,'market_min'),
(12906,703,'market_25'),
(23248,1123,'market_min'),
(23248,1125,'market_min'),
(4720,703,'market_min'),
(3664,1123,'market_25'),
(6952,1123,'market_min'),
(11618,702,'market_min'),
(23230,701,'market_min'),
(6968,701,'market_25'),
(6968,704,'market_min'),
(11059,703,'market_min'),
(11059,704,'market_min'),
(12430,704,'market_min'),
(12041,701,'market_25'),
(22327,703,'market_min'),
(23032,701,'market_min'),
(23032,703,'market_min'),
(23032,704,'market_min'),
(2421,703,'market_min'),
(6965,701,'market_25'),
(6965,703,'market_min'),
(5869,701,'market_min'),
(12608,1126,'market_min'),
(26028,703,'market_25'),
(3850,701,'market_min'),
(12539,703,'market_25'),
(3851,701,'market_min'),
(3851,704,'market_min'),
(3849,701,'market_min'),
(27219,701,'market_min'),
(27219,703,'market_min'),
(5410,703,'market_min'),
(22665,701,'market_min'),
(25288,700,'market_25'),
(24566,1123,'market_min'),
(26387,704,'market_25'),
(10940,704,'market_min'),
(8023,704,'market_min'),
(687,700,'market_25'),
(3057,703,'market_25'),
(22865,703,'market_25'),
(9483,703,'market_min'),
(2013,701,'market_25'),
(22871,703,'market_min'),
(10722,701,'market_min'),
(10722,703,'market_min'),
(10722,704,'market_min'),
(22921,704,'market_min'),
(13487,704,'market_25'),
(12994,701,'market_25'),
(12994,703,'market_min'),
(23237,1126,'market_min'),
(11857,701,'market_min'),
(11857,1125,'market_min'),
(2367,703,'market_25'),
(22049,701,'market_min'),
(23488,703,'market_min'),
(10146,703,'market_25'),
(7191,1126,'market_min'),
(24669,701,'market_min'),
(24669,704,'market_min'),
(24669,1123,'market_min'),
(1742,701,'market_min'),
(11930,704,'market_25'),
(9922,704,'market_25'),
(25994,704,'market_min'),
(807,700,'market_25'),
(807,701,'market_min'),
(3865,701,'market_25'),
(2731,701,'market_min'),
(9484,700,'market_25'),
(20220,703,'market_min'),
(10535,704,'market_min'),
(11659,703,'market_25'),
(2318,701,'market_min'),
(12919,701,'market_min'),
(23602,701,'market_min'),
(23974,700,'market_min'),
(22362,701,'market_min'),
(22362,703,'market_25'),
(22362,1124,'market_min'),
(21684,703,'market_min'),
(8503,701,'market_25'),
(23660,701,'market_min'),
(23660,1124,'market_25'),
(13192,701,'market_min'),
(13192,704,'market_min'),
(334,702,'market_min'),
(334,703,'market_25'),
(334,704,'market_25'),
(12623,702,'market_min'),
(12623,703,'market_25'),
(5408,700,'market_min'),
(5408,704,'market_min'),
(24246,701,'market_min'),
(5409,703,'market_min'),
(21480,703,'market_min'),
(23857,701,'market_min'),
(1294,701,'market_min'),
(24668,703,'market_min'),
(24668,704,'market_25'),
(24668,1126,'market_min'),
(11092,704,'market_25'),
(6556,701,'market_min'),
(4071,700,'market_min'),
(3584,700,'market_min'),
(11290,703,'market_25'),
(11290,1126,'market_min'),
(22361,701,'market_min'),
(22361,702,'market_min'),
(22361,1124,'market_min'),
(23232,701,'market_min'),
(13112,704,'market_25'),
(9480,701,'market_min'),
(9480,704,'market_min'),
(11226,1126,'market_min'),
(21810,704,'market_min'),
(8364,701,'market_min'),
(5723,703,'market_25'),
(3970,701,'market_min'),
(11859,1125,'market_min'),
(1447,703,'market_min'),
(9531,704,'market_25'),
(12913,703,'market_25'),
(12513,701,'market_min'),
(22324,703,'market_min'),
(22324,1123,'market_25'),
(4866,700,'market_min'),
(4866,701,'market_min'),
(21806,704,'market_min'),
(22572,704,'market_min'),
(11358,701,'market_min'),
(13214,704,'market_min'),
(12918,701,'market_min'),
(1560,702,'market_min'),
(8923,1125,'market_min'),
(1513,704,'market_min'),
(9373,1125,'market_min'),
(6030,701,'market_min'),
(22667,704,'market_min'),
(12419,701,'market_min'),
(12419,704,'market_min'),
(9481,701,'market_25'),
(11053,701,'market_min'),
(25766,704,'market_min'),
(22967,701,'market_25'),
(737,701,'market_min'),
(12048,703,'market_min'),
(12048,704,'market_min'),
(12048,1124,'market_min'),
(22364,700,'market_min'),
(3453,701,'market_min'),
(25768,703,'market_25'),
(2661,703,'market_25'),
(6946,703,'market_25'),
(5517,704,'market_min'),
(25769,703,'market_min'),
(12040,701,'market_25'),
(12040,704,'market_25'),
(8504,701,'market_25'),
(689,701,'market_min'),
(20984,703,'market_min'),
(2480,701,'market_min'),
(11369,701,'market_min'),
(13357,701,'market_25'),
(23662,700,'market_25'),
(23662,1124,'market_min'),
(11043,701,'market_min'),
(10942,703,'market_min'),
(13857,703,'market_min'),
(3396,700,'market_min'),
(12598,704,'market_min'),
(515,702,'market_min'),
(7568,700,'market_25'),
(25765,700,'market_25'),
(25765,703,'market_25'),
(5603,701,'market_min'),
(26443,701,'market_25'),
(23858,700,'market_min'),
(11858,701,'market_min'),
(11858,1125,'market_min'),
(20223,704,'market_25'),
(5642,703,'market_min'),
(24384,701,'market_min'),
(12257,701,'market_min'),
(22363,703,'market_25'),
(448,700,'market_min'),
(26594,701,'market_min'),
(12920,703,'market_25'),
(24626,704,'market_min'),
(10534,703,'market_25'),
(2895,703,'market_min'),
(11528,1125,'market_min'),
(14034,701,'market_min'),
(13254,1123,'market_min'),
(11044,703,'market_25'),
(12253,704,'market_min'),
(12658,703,'market_min'),
(10900,700,'market_25'),
(5604,1125,'market_min'),
(26445,701,'market_25'),
(9382,700,'market_min'),
(23855,701,'market_min'),
(5602,700,'market_min'),
(26449,701,'market_25'),
(26449,703,'market_25'),
(26449,704,'market_25'),
(9186,701,'market_25'),
(23843,701,'market_min'),
(21081,704,'market_min'),
(24228,703,'market_25'),
(24228,704,'market_25'),
(20582,701,'market_25'),
(21165,700,'market_min'),
(22874,700,'market_25'),
(1205,703,'market_min'),
(12256,703,'market_min'),
(11625,1125,'market_min'),
(12665,701,'market_min'),
(27204,701,'market_min'),
(27206,701,'market_25'),
(27208,701,'market_min'),
(27261,701,'market_25'),
(11212,703,'market_min'),
(23684,701,'market_min')
]


# =============================================================================
# COMPUTE NEW PRICES (per product × cohort)
# =============================================================================
ACTION_TO_COLUMN = {
    'market_min': 'market_min',
    'market_25': 'market_25',
    'market_50': 'market_50',
    'market_75': 'market_75',
    'market_max': 'market_max',
    'market_avg': 'market_avg',
}

results = []
for product_id,cohort_id, action in PUSH_LIST:
    product_rows = lookup[(lookup['product_id'] == product_id)&(lookup['cohort_id'] == cohort_id)] #
    if product_rows.empty:
        results.append({'product_id': product_id, 'cohort_id': '-', 'action': str(action), 'status': 'NOT FOUND'})
        continue

    for _, row in product_rows.iterrows():
        cohort_id = int(row['cohort_id'])
        wac = row.get('wac_p', None)
        sku = row.get('sku', '')
        brand = row.get('brand', '')

        if isinstance(action, (int, float)):
            base_price = float(action)
            action_label = f'fixed_{action}'
        elif action == 'target_margin':
            tm = row.get('target_margin', 0.05)
            if pd.isna(wac) or wac <= 0:
                results.append({'product_id': product_id, 'cohort_id': cohort_id, 'sku': sku, 'action': action, 'status': 'NO WAC'})
                continue
            base_price = wac / (1 - tm)
            action_label = f'target_margin ({tm:.1%})'
        elif action == 'step_up':
            tiers = row.get('price_tiers', [])
            cur = row.get('current_price', 0)
            if not tiers or pd.isna(cur) or cur <= 0:
                results.append({'product_id': product_id, 'cohort_id': cohort_id, 'sku': sku, 'action': action, 'status': 'NO TIERS OR PRICE'})
                continue
            next_up = [t for t in tiers if t > cur + 0.25]
            if not next_up:
                results.append({'product_id': product_id, 'cohort_id': cohort_id, 'sku': sku, 'action': action, 'status': 'ALREADY AT TOP'})
                continue
            base_price = next_up[0]
            action_label = f'step_up ({cur:.2f} -> {base_price:.2f})'
        elif action == 'step_down':
            tiers = row.get('price_tiers', [])
            cur = row.get('current_price', 0)
            if not tiers or pd.isna(cur) or cur <= 0:
                results.append({'product_id': product_id, 'cohort_id': cohort_id, 'sku': sku, 'action': action, 'status': 'NO TIERS OR PRICE'})
                continue
            next_down = [t for t in reversed(tiers) if t < cur - 0.5]
            if not next_down:
                results.append({'product_id': product_id, 'cohort_id': cohort_id, 'sku': sku, 'action': action, 'status': 'ALREADY AT BOTTOM'})
                continue
            base_price = next_down[0]
            action_label = f'step_down ({cur:.2f} -> {base_price:.2f})'
        elif action in ACTION_TO_COLUMN:
            col = ACTION_TO_COLUMN[action]
            val = row.get(col, None)
            if pd.isna(val):
                results.append({'product_id': product_id, 'cohort_id': cohort_id, 'sku': sku, 'action': action, 'status': 'NO MARKET DATA'})
                continue
            base_price = val
            action_label = action
        else:
            results.append({'product_id': product_id, 'cohort_id': cohort_id, 'sku': sku, 'action': str(action), 'status': 'INVALID ACTION'})
            continue

        base_price_rounded = np.round(base_price * 4) / 4
        margin = (base_price_rounded - wac) / base_price_rounded if (wac and wac > 0 and base_price_rounded > 0) else None

        cur_price = row.get('current_price', None)

        results.append({
            'product_id': product_id,
            'cohort_id': cohort_id,
            'sku': sku,
            'brand': brand,
            'action': action_label,
            'wac': wac,
            'current_price': cur_price,
            'base_price': base_price,
            'new_price': base_price_rounded,
            'margin': margin,
            'status': 'OK',
        })

df_results = pd.DataFrame(results)

ok_count = (df_results['status'] == 'OK').sum()
err_count = len(df_results) - ok_count
products_ok = df_results[df_results['status'] == 'OK']['product_id'].nunique()
print(f'Results: {ok_count} OK across {products_ok} products × cohorts, {err_count} errors/skipped')
if err_count > 0:
    print('\nErrors/Skipped:')
    print(df_results[df_results['status'] != 'OK'][['product_id', 'cohort_id', 'sku', 'action', 'status']].to_string(index=False))
df_results[df_results['status'] == 'OK']

Results: 570 OK across 417 products × cohorts, 0 errors/skipped


,product_id,cohort_id,sku,brand,action,wac,current_price,base_price,new_price,margin,status
0,3985,704,رويال أعشاب ليمون جنزبيل - 50 فتلة,رويال,fixed_36.8025457936824,34.962419,37.50,36.802546,36.75,0.048642,OK
1,25698,701,كوكى مرقة شريط 8 جرام - 48 مكعب,كوكى مرقة,fixed_35.3815682215049,33.612490,36.50,35.381568,35.50,0.053169,OK
2,6779,700,فريش فارم كفتة بقرى - 350 جم,فريش فارم,fixed_81.0789473684211,77.025000,82.25,81.078947,81.00,0.049074,OK
3,24242,1124,كيك تاوتاو تربو كاكاو بكريمة الفانيليا - 6 قطع...,تاوتاو,fixed_22.8100016931385,21.669502,25.00,22.810002,22.75,0.047494,OK
4,13598,1124,ليندو برو برائحه اللافندر - 120 جرام,ليندو,fixed_84.8684210526316,80.625000,90.25,84.868421,84.75,0.048673,OK
...,...,...,...,...,...,...,...,...,...,...,...
565,27206,701,زينه مطبخ مضغوط (12 كيس *2 بكرة) - 2 بكرة,زينه,market_25,321.742800,378.25,347.750000,347.75,0.074787,OK
566,27208,701,(6بكره) XXL زينه رول متعدد الاستخدام - 6 بكرة,زينه,market_min,548.386200,641.75,586.500000,586.50,0.064985,OK
567,27261,701,زينه متعدد الاستخدام لارج - 6 بكرة,زينه,market_25,321.498000,376.25,342.500000,342.50,0.061320,OK
568,11212,703,فاين فودز مرقة خضار 8 مكعب عرض (7+1) - 72 جم,كنور,market_min,116.450052,132.50,122.000000,122.00,0.045491,OK


In [17]:
df_results=df_results[df_results['current_price']>df_results['new_price']]
df_results['change'] = (df_results['new_price'] - df_results['current_price'])/df_results['current_price']
print(df_results['margin'].mean())
print(df_results['margin'].median())
print(df_results['margin'].min())
print(df_results['margin'].max())
#df_results=df_results[~df_results['brand'].isin(['كوكا كولا','شويبس'])]
df_results

0.05561264224382602
0.054960234399808544
-0.10815446773220401
0.18173476857029308


,product_id,cohort_id,sku,brand,action,wac,current_price,base_price,new_price,margin,status,change
0,3985,704,رويال أعشاب ليمون جنزبيل - 50 فتلة,رويال,fixed_36.8025457936824,34.962419,37.50,36.802546,36.75,0.048642,OK,-0.020000
1,25698,701,كوكى مرقة شريط 8 جرام - 48 مكعب,كوكى مرقة,fixed_35.3815682215049,33.612490,36.50,35.381568,35.50,0.053169,OK,-0.027397
2,6779,700,فريش فارم كفتة بقرى - 350 جم,فريش فارم,fixed_81.0789473684211,77.025000,82.25,81.078947,81.00,0.049074,OK,-0.015198
3,24242,1124,كيك تاوتاو تربو كاكاو بكريمة الفانيليا - 6 قطع...,تاوتاو,fixed_22.8100016931385,21.669502,25.00,22.810002,22.75,0.047494,OK,-0.090000
4,13598,1124,ليندو برو برائحه اللافندر - 120 جرام,ليندو,fixed_84.8684210526316,80.625000,90.25,84.868421,84.75,0.048673,OK,-0.060942
...,...,...,...,...,...,...,...,...,...,...,...,...
565,27206,701,زينه مطبخ مضغوط (12 كيس *2 بكرة) - 2 بكرة,زينه,market_25,321.742800,378.25,347.750000,347.75,0.074787,OK,-0.080635
566,27208,701,(6بكره) XXL زينه رول متعدد الاستخدام - 6 بكرة,زينه,market_min,548.386200,641.75,586.500000,586.50,0.064985,OK,-0.086093
567,27261,701,زينه متعدد الاستخدام لارج - 6 بكرة,زينه,market_25,321.498000,376.25,342.500000,342.50,0.061320,OK,-0.089701
568,11212,703,فاين فودز مرقة خضار 8 مكعب عرض (7+1) - 72 جم,كنور,market_min,116.450052,132.50,122.000000,122.00,0.045491,OK,-0.079245


In [18]:
mona_query = '''
with base as (
select  dt.taggable_id as retailer_id,
		dynamic_tags.name as dynamic_tag_name, 
		dynamic_tags.id as dynamic_tag_id, 
		c.id as new_cohort_id, 
		c.name as new_cohort_name
from dynamic_tags
inner join dynamic_taggables dt on dt.dynamic_tag_id = dynamic_tags.id
left join cohorts c on c.dynamic_tag_id = dynamic_tags.id
where dynamic_tags.name in ('mona_700', 'mona_701', 'mona_702', 'mona_703', 'mona_704', 'mona_1123', 'mona_1124', 'mona_1125', 'mona_1126', 'mona_cairo')), 

retailer_cohorts as (
select * 
from (
select distinct	
	taggable_id as retailer_id ,
	case when custom_ct =1 then cohort_id end as custom_cohort,
	case when custom_ct = 1 then fallbk else cohort_id end as main_cohort,
	61 as general_cohort, 
	dynamic_tag_name, 
	new_cohort_id, 
	new_cohort_name
from (
select 
	taggable_id,
	cohorts.id as cohort_id,
	FALLBACK_COHORT_ID as fallbk,
	cohorts.priority,
	case when coalesce(FALLBACK_COHORT_ID,61) = 61 then 0 else 1 end as custom_ct,
	rank () over (partition by taggable_id,custom_ct order by priority) as rk,
	sum (custom_ct) over (partition by taggable_id) as custom_count, 
	dynamic_tag_name, 
	new_cohort_id, 
	new_cohort_name
from cohorts
	left join dynamic_taggables as dt on dt.dynamic_tag_id  = cohorts.dynamic_tag_id
	inner join base on base.retailer_id = dt.taggable_id 
where cohorts.is_active = true and cohorts.id <> 61 
) as sub 
where rk = 1 and (custom_count = 0 or (custom_count >0 and custom_ct = 1))
order by 1)), 

served_warehouses as (
select  warehouse_id, 
		new_cohort_id, 
		count(distinct parent_sales_order_id) order_counts
from sales_orders 
inner join retailer_cohorts on retailer_cohorts.retailer_id = sales_orders.retailer_id 
where sales_orders.created_at::date between current_date - 90 and current_date 
group by all ), 

historical_stocks as (
select  distinct product_id, new_cohort_id
from materialized_views.stock_day_close std
inner join served_warehouses sw on sw.warehouse_id = std.warehouse_id 
where available_stock > 1
and timestamp::date between current_date - 30 and current_date), 

stock_now as (
select distinct product_id, new_cohort_id
from product_warehouse 
inner join served_warehouses sw on sw.warehouse_id = product_warehouse.warehouse_id 
where available_stock > 0), 

cohort_fallbacks as (
select  distinct custom_cohort, 
		main_cohort, 
		general_cohort, 
		new_cohort_id, 
		new_cohort_name
from retailer_cohorts), 

final as (
select  product_id as "Product ID", 
		product_name as "Product Name", 
		packing_unit_id as "Packing Unit ID", 
		price as "Price", 
		case when visibility = 1 then 'YES' else 'NO' end as "Visibility (YES/NO)", 
		null as "Execute At (format:dd/mm/yyyy HH:mm)", 
		null as "Tags", 
		new_cohort_id, 
		new_cohort_name
from (
select s.*,
	   coalesce(case when cppu3.is_customized = true then  cppu3.price end, cppu2.price,cppu.price) as price, 
	   case when c.section_id in (714,626, 516, 417, 351,121,285,20,54,37,10,43,44,36,14,8,55,619,622,621) and (stock_now.product_id is not null or historical_stocks.product_id is not null) then 1 else 0 end as visibility, 
	   p.name_ar as product_name
from (
select product_id, 
       packing_unit_id, 
	   custom_cohort, 
	   main_cohort, 
	   general_cohort, 
	   new_cohort_id, 
	   new_cohort_name, 
	   packing_unit_products.id as product_packing_unit_id
from packing_unit_products
cross join cohort_fallbacks
where DELETEd_AT is null
) s 
left join products p on p.id = s.product_id 
left join categories c on c.id = p.category_id 
left join cohort_product_packing_units as cppu on cppu.cohort_id = s.general_cohort and cppu.product_packing_unit_id = s.product_packing_unit_id
left join cohort_product_packing_units as cppu2 on cppu2.cohort_id = s.main_cohort and cppu2.product_packing_unit_id = s.product_packing_unit_id
left join cohort_product_packing_units as cppu3 on cppu3.cohort_id = s.custom_cohort and cppu3.product_packing_unit_id = s.product_packing_unit_id
left join historical_stocks on historical_stocks.product_id = s.product_id and historical_stocks.new_cohort_id = s.new_cohort_id
left join stock_now on stock_now.product_id = s.product_id and stock_now.new_cohort_id = s.new_cohort_id
)) 


select final.*,
       cppu.price AS db_price
from final
join packing_unit_products pup
  on pup.product_id = final."Product ID"
 and pup.packing_unit_id = final."Packing Unit ID"
left join cohort_product_packing_units as cppu
  on cppu.cohort_id = final.new_cohort_id
 and cppu.product_packing_unit_id = pup.id
where cppu.price <> final."Price"
'''
#mona_data = query_snowflake(mona_query)
#len(mona_data)

In [19]:
# for cohort_id in mona_data.NEW_COHORT_ID.unique():
#     out = mona_data[mona_data['NEW_COHORT_ID'] == cohort_id]
#     id_ = cohort_id
#     name = out.NEW_COHORT_NAME.values[0]
#     file_name_ = 'uploads/1_new_{}.xlsx'.format(name).replace(' ','_')
#     out.to_excel(file_name_,index = False)
#     ################### Loop to avoid file limit ######################
#     # split file into chunks
#     print('Spliting file into chunks...')
#     if id_ == 61:
#         chunks = [out[i:i + 2000] for i in range(0, len(out), 2000)]
#     else:
#         chunks = [out[i:i + 4000] for i in range(0, len(out), 4000)]
#     print(f'len chunks = {len(chunks)}')
#     fileslist = []
#     for i, chunk in tqdm(enumerate(chunks), total=len(chunks)):
#         fileslist.append(f'manual/output_{id_}_chunk_{i + 1}.xlsx')
#         output_file_path = f'manual/output_{id_}_chunk_{i + 1}.xlsx'
#         chunk.to_excel(output_file_path, index=False, engine='xlsxwriter')
#     # Loop over chunks and upload
#     print('Uploading...')
#     for file in fileslist:
#         chunk = file.split('chunk_')[1].split('.xls')[0]
#         x = post_prices(id_, file)
#         # print(str(x.content))
#         if ('"success":true' in str(x.content).lower()):
#             print(f"Prices are upoladed successfuly Region: {name} ,cohort: {id_}, chunk: {chunk}")
#         else:
#             print(f"ERROR Region: {name} ,cohort: {id_}, chunk: {chunk}")
#             print(x.content)
#             final_status = False
#             break

In [20]:
# =============================================================================
# PUSH PRICES
# =============================================================================
MODE = 'live'  # Change to 'live' to actually push

df_ok = df_results[df_results['status'] == 'OK'].copy()

# Filter to only SKUs with stock
if len(df_ok) > 0:
    stock_by_product = df_stocks.groupby('product_id')['stocks'].sum().reset_index()
    df_ok = df_ok.merge(stock_by_product, on='product_id', how='left')
    no_stock = df_ok['stocks'].fillna(0) <= 0
    if no_stock.any():
        print(f'Skipping {no_stock.sum()} rows with no stock')
        df_ok = df_ok[~no_stock].copy()

if len(df_ok) == 0:
    print('No valid prices to push.')
else:
    wh_df = pd.DataFrame(WAREHOUSE_MAPPING, columns=['region', 'warehouse', 'warehouse_id', 'cohort_id'])

    push_rows = []
    for _, r in df_ok.iterrows():
        target_cohort = int(r['cohort_id'])
        cohort_whs = wh_df[wh_df['cohort_id'] == target_cohort]

        if cohort_whs.empty:
            cohort_whs = pd.DataFrame([{'warehouse_id': 1, 'cohort_id': target_cohort}])

        cur_price = r['current_price'] if pd.notna(r.get('current_price')) else 0

        for _, wh in cohort_whs.iterrows():
            push_rows.append({
                'product_id': int(r['product_id']),
                'sku': r['sku'],
                'new_price': r['new_price'],
                'warehouse_id': int(wh['warehouse_id']),
                'cohort_id': target_cohort,
                'stocks': 1,
                'current_price': cur_price,
            })

    df_push = pd.DataFrame(push_rows)

    n_products = df_ok['product_id'].nunique()
    n_cohorts = df_ok['cohort_id'].nunique()
    print(f'Pushing {n_products} products × {n_cohorts} cohorts = {len(df_push)} rows')
    print(f'Mode: {MODE}')
    print()

    result = push_prices(
        df_prices=df_push,
        pus=df_pus,
        source_module='manual_push',
        mode=MODE,
    )
    print(f'\nDone. Result: {result}')

Pushing 417 products × 9 cohorts = 939 rows
Mode: live


🚀 MODE: LIVE
   Files will be prepared AND uploaded to API
Loading disable_pu_visibility from Google Sheets...
  ✓ Loaded 88 products to disable min PU visibility

PUSH PRICES - Source: manual_push
Total received: 939
Price changes to push: 939
Skipped (no change): 0

Price changes summary:
  Increases: 0
  Decreases: 939

🔗 Mirrored prices to 6 main/general cohorts (+642 rows)
   Cohort 695 ← 700: 98 rows
   Cohort 61 ← 700: 98 rows
   Cohort 699 ← 702: 48 rows
   Cohort 697 ← 703: 185 rows
   Cohort 698 ← 704: 184 rows
   Cohort 696 ← 1123: 29 rows

📋 Prepared 1550 packing unit prices (incl. main cohorts)

Processing cohort: 61
  Saved: uploads/manual_push_61.xlsx (98 rows)
  Split into 1 chunks (size: 2000)


  Saving chunks: 100%|██████████| 1/1 [00:00<00:00, 94.97it/s]

  Uploading...


    ✓ Chunk 1 uploaded successfully

Processing cohort: 695
  Saved: uploads/manual_push_695.xlsx (98 rows)
  Split into 1 chunks (size: 4000)


  Saving chunks: 100%|██████████| 1/1 [00:00<00:00, 95.85it/s]

...

  Uploading...


    ✓ Chunk 1 uploaded successfully

Processing cohort: 1126
  Saved: uploads/manual_push_1126.xlsx (43 rows)
  Split into 1 chunks (size: 4000)


  Saving chunks: 100%|██████████| 1/1 [00:00<00:00, 138.03it/s]

  Uploading...


    ✓ Chunk 1 uploaded successfully

🚀 UPLOAD COMPLETE
Mode: live
Total prepared: 1550
Total failed: 0
  Cleanup: removed 30 temporary .xlsx files from 2 folder(s)

Done. Result: {'total_received': 939, 'price_changes': 939, 'pushed': 1550, 'failed': 0, 'skipped': 0, 'source_module': 'manual_push', 'timestamp': '2026-04-27 15:28:35', 'mode': 'live', 'skipped_cohorts': []}
